# PM System · 의사결정 보조 대시보드

**런타임 → 모두 실행** 후 `별도 탭에서 대시보드 열기`를 누르세요. CPU만 있으면 됩니다.

- 프로젝트 제목·목적·성공 조건·기간 편집
- 목표·작업·결정 요청 추가 / 수정 / 삭제
- 실제 날짜에 비례한 가지형 타임라인, 드래그 이동, 확대·축소
- 자동 파일 저장, JSON 백업 저장·불러오기

기본 실행에는 패키지를 설치하지 않습니다. 선택 셀에서 공개 링크를 켜는 경우에만 Colab에 Gradio를 설치합니다.
**코랩은 상시 서버가 아닙니다. 런타임 초기화 시 저장 파일도 사라질 수 있으므로 백업을 내려받으세요.**
실제 운영 데이터가 포함된 실행 출력·백업은 공개 GitHub에 올리지 마세요.


## 1. 프로그램 준비
소스가 포함되어 있어 이 노트북 파일만으로 실행할 수 있습니다.


In [ ]:
from pathlib import Path
import json, sys

APP_DIR = Path('/content/PM_system_dashboard')
# 이 노트북에 소스를 포함하므로 저장소 연결이나 API 키 없이 실행됩니다.
FILES = json.loads('{"dashboard/__init__.py": "\\"\\"\\"의사결정 지원 대시보드.\\"\\"\\"\\n", "dashboard/server.py": "\\"\\"\\"표준 라이브러리만으로 실행하는 대시보드 서버.\\n\\npython -m dashboard.server\\n운영 자료는 Git에 포함하지 않는 data/private/dashboard.json에 저장한다.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport copy\\nimport json\\nimport os\\nimport re\\nimport secrets\\nimport threading\\nfrom datetime import date, timedelta\\nfrom http.cookies import SimpleCookie\\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\\nfrom pathlib import Path\\nfrom urllib.parse import parse_qs, urlsplit\\n\\nROOT = Path(__file__).resolve().parent\\nSTATUSES = (\'예정\', \'진행 중\', \'완료\', \'차단\')\\n# 앱 설치(PWA)에 필요한 화면 파일. 운영 데이터는 포함하지 않는다.\\nPUBLIC_FILES = {\'manifest.webmanifest\': \'application/manifest+json; charset=utf-8\',\\n                \'sw.js\': \'text/javascript; charset=utf-8\',\\n                \'icon.svg\': \'image/svg+xml\'}\\n\\n\\ndef initial_state():\\n    today = date.today()\\n    d = lambda n: (today + timedelta(days=n)).isoformat()\\n    return {\'version\': 1, \'revision\': 0,\\n            \'project\': {\'title\': \'새 프로젝트\', \'purpose\': \'팀이 달성하려는 목적을 작성하세요.\',\\n                        \'success\': \'완료 여부를 확인할 기준을 작성하세요.\',\\n                        \'start\': d(-7), \'end\': d(21)},\\n            \'goals\': [], \'tasks\': [], \'decisions\': []}\\n\\n\\ndef validate(state):\\n    \\"\\"\\"가져오기와 저장 모두 같은 규칙을 적용한다.\\"\\"\\"\\n    if not isinstance(state, dict) or state.get(\'version\') != 1:\\n        raise ValueError(\'지원하는 대시보드 백업 파일(version 1)이 아닙니다.\')\\n    def text(obj, key, required=False, limit=3000):\\n        value = obj.get(key, \'\')\\n        if not isinstance(value, str) or len(value) > limit:\\n            raise ValueError(f\'{key}: 문자열 길이가 올바르지 않습니다.\')\\n        if required and not value.strip():\\n            raise ValueError(\'제목은 비워둘 수 없습니다.\')\\n    def dates(obj, start=\'start\', end=\'end\'):\\n        a, b = date.fromisoformat(obj[start]), date.fromisoformat(obj[end])\\n        if a > b or (b-a).days > 3650:\\n            raise ValueError(\'종료일은 시작일 이후이며 기간은 10년 이내여야 합니다.\')\\n    p = state[\'project\']\\n    for k in (\'title\', \'purpose\', \'success\'):\\n        text(p, k, k == \'title\')\\n    dates(p)\\n    ids = set()\\n    for group in (\'goals\', \'tasks\', \'decisions\'):\\n        if not isinstance(state.get(group), list) or len(state[group]) > 500:\\n            raise ValueError(\'항목은 종류별 500개까지 저장할 수 있습니다.\')\\n        for item in state[group]:\\n            if not isinstance(item, dict):\\n                raise ValueError(\'항목 형식이 올바르지 않습니다.\')\\n            text(item, \'id\', True, 100)\\n            if item[\'id\'] in ids:\\n                raise ValueError(\'중복된 항목 ID입니다.\')\\n            ids.add(item[\'id\'])\\n            text(item, \'title\', True, 200)\\n            text(item, \'note\')\\n            if group == \'tasks\':\\n                dates(item)\\n                text(item, \'owner\', limit=100)\\n                if item.get(\'status\') not in STATUSES:\\n                    raise ValueError(\'작업 상태가 올바르지 않습니다.\')\\n            else:\\n                date.fromisoformat(item[\'due\'])\\n                if not isinstance(item.get(\'done\'), bool):\\n                    raise ValueError(\'완료 상태가 올바르지 않습니다.\')\\n    tasks = {t[\'id\']: t for t in state[\'tasks\']}\\n    horizon = [date.fromisoformat(p[\'start\']), date.fromisoformat(p[\'end\'])]\\n    horizon.extend(date.fromisoformat(t[k]) for t in tasks.values() for k in (\'start\',\'end\'))\\n    if (max(horizon)-min(horizon)).days > 3650:\\n        raise ValueError(\'프로젝트와 작업의 전체 시간 범위는 10년 이내여야 합니다.\')\\n    for t in tasks.values():\\n        seen = {t[\'id\']}\\n        pred = t.get(\'predecessor\', \'\')\\n        while pred:\\n            if pred not in tasks or pred in seen:\\n                raise ValueError(\'선행 작업이 없거나 순환 관계입니다.\')\\n            seen.add(pred)\\n            pred = tasks[pred].get(\'predecessor\', \'\')\\n        pred = t.get(\'predecessor\', \'\')\\n        if pred and t[\'start\'] < tasks[pred][\'end\']:\\n            raise ValueError(\'후속 작업 시작일은 선행 작업 종료일 이후여야 합니다.\')\\n        goal = t.get(\'goal\', \'\')\\n        if goal and goal not in {g[\'id\'] for g in state[\'goals\']}:\\n            raise ValueError(\'연결된 목표가 없습니다.\')\\n    # 반환할 필드만 유지해서 가져온 파일의 불필요한 데이터는 저장하지 않는다.\\n    result = {\'version\': 1, \'project\': {k: p[k] for k in (\'title\',\'purpose\',\'success\',\'start\',\'end\')}}\\n    fields = {\'goals\': (\'id\',\'title\',\'due\',\'note\',\'done\'),\\n              \'decisions\': (\'id\',\'title\',\'due\',\'note\',\'done\'),\\n              \'tasks\': (\'id\',\'title\',\'start\',\'end\',\'owner\',\'status\',\'note\',\'predecessor\',\'goal\')}\\n    for group, keys in fields.items():\\n        result[group] = [{k: item.get(k, \'\') for k in keys} for item in state[group]]\\n    return result\\n\\n\\nclass Store:\\n    def __init__(self, path):\\n        self.path = Path(path)\\n        self.lock = threading.RLock()\\n        self.state = initial_state()\\n        if self.path.exists():\\n            raw = json.loads(self.path.read_text(encoding=\'utf-8\'))\\n            self.state = validate(raw)\\n            self.state[\'revision\'] = int(raw.get(\'revision\', 0))\\n\\n    def read(self):\\n        with self.lock:\\n            return copy.deepcopy(self.state)\\n\\n    def save(self, state, revision):\\n        with self.lock:\\n            if revision != self.state[\'revision\']:\\n                raise RuntimeError(\'다른 창에서 내용이 바뀌었습니다. 새로고침 후 다시 수정하세요.\')\\n            clean = validate(state)\\n            clean[\'revision\'] = self.state[\'revision\'] + 1\\n            self.path.parent.mkdir(parents=True, exist_ok=True)\\n            tmp = self.path.with_suffix(\'.tmp\')\\n            tmp.write_text(json.dumps(clean, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n            os.replace(tmp, self.path)\\n            self.state = clean\\n            return self.read()\\n\\n\\nPROJECT_ID = re.compile(r\'^[A-Za-z0-9_-]{1,64}$\')\\nMAX_PROJECTS = 50\\n\\n\\nclass Projects:\\n    \\"\\"\\"프로젝트마다 파일 하나(projects/<id>.json)로 저장한다.\\n\\n    이전 버전의 단일 파일(dashboard.json)이 있고 프로젝트가 없으면 첫 프로젝트로 옮긴다.\\n    원본 파일은 지우지 않는다.\\n    \\"\\"\\"\\n    def __init__(self, directory, legacy=None):\\n        self.dir = Path(directory)\\n        self.lock = threading.RLock()\\n        self.stores = {}\\n        if self.dir.exists():\\n            for f in sorted(self.dir.glob(\'*.json\')):\\n                if PROJECT_ID.match(f.stem):\\n                    self.stores[f.stem] = Store(f)\\n        if not self.stores and legacy and Path(legacy).exists():\\n            pid = secrets.token_hex(6)\\n            self.dir.mkdir(parents=True, exist_ok=True)\\n            dest = self.dir/f\'{pid}.json\'\\n            dest.write_text(Path(legacy).read_text(encoding=\'utf-8\'), encoding=\'utf-8\')\\n            self.stores[pid] = Store(dest)\\n\\n    def list(self):\\n        with self.lock:\\n            items = [{\'id\': pid, \'state\': store.read()} for pid, store in self.stores.items()]\\n        return sorted(items, key=lambda x: (x[\'state\'][\'project\'][\'start\'], x[\'state\'][\'project\'][\'title\'], x[\'id\']))\\n\\n    def get(self, pid):\\n        with self.lock:\\n            if not PROJECT_ID.match(pid or \'\') or pid not in self.stores:\\n                raise LookupError(\'프로젝트를 찾을 수 없습니다. 목록에서 다시 선택하세요.\')\\n            return self.stores[pid]\\n\\n    def create(self, title=\'\'):\\n        with self.lock:\\n            if len(self.stores) >= MAX_PROJECTS:\\n                raise ValueError(f\'프로젝트는 {MAX_PROJECTS}개까지 만들 수 있습니다.\')\\n            state = initial_state()\\n            if title:\\n                state[\'project\'][\'title\'] = title\\n            pid = secrets.token_hex(6)\\n            store = Store(self.dir/f\'{pid}.json\')\\n            saved = store.save(state, 0)\\n            self.stores[pid] = store\\n            return {\'id\': pid, \'state\': saved}\\n\\n\\nAPI = re.compile(r\'/api/projects(?:/([^/]+)/(state|export))?$\')\\n\\n\\ndef start_server(port=8765, data_path=None, token=None):\\n    legacy = Path(data_path or ROOT.parent/\'data/private/dashboard.json\')\\n    projects = Projects(legacy.parent/\'projects\', legacy)\\n    token = token or secrets.token_urlsafe(24)\\n    class Handler(BaseHTTPRequestHandler):\\n        def log_message(self, *_):\\n            pass  # 접속 키와 실제 운영 정보를 로그에 남기지 않는다.\\n\\n        def reply(self, code, body, content_type=\'application/json; charset=utf-8\', headers=None):\\n            if not isinstance(body, bytes):\\n                body = json.dumps(body, ensure_ascii=False).encode(\'utf-8\')\\n            self.send_response(code)\\n            self.send_header(\'Content-Type\', content_type)\\n            self.send_header(\'Cache-Control\', \'no-store\')\\n            self.send_header(\'X-Content-Type-Options\', \'nosniff\')\\n            for key, value in (headers or {}).items():\\n                self.send_header(key, value)\\n            self.send_header(\'Content-Length\', str(len(body)))\\n            self.end_headers()\\n            self.wfile.write(body)\\n\\n        def authorized(self):\\n            cookies = SimpleCookie()\\n            try:\\n                cookies.load(self.headers.get(\'Cookie\', \'\'))\\n                candidate = cookies[\'pm_access\'].value if \'pm_access\' in cookies else \'\'\\n            except Exception:\\n                candidate = \'\'\\n            return secrets.compare_digest(candidate, token)\\n\\n        def do_GET(self):\\n            url = urlsplit(self.path)\\n            key = parse_qs(url.query).get(\'key\', [\'\'])[0]\\n            if key and secrets.compare_digest(key, token):\\n                self.reply(303, b\'\', headers={\'Location\': \'./\', \'Set-Cookie\':\\n                    f\'pm_access={token}; Path=/; HttpOnly; SameSite=Lax\'})\\n                return\\n            name = url.path.rsplit(\'/\', 1)[-1]\\n            if name in PUBLIC_FILES:\\n                # 설치형 앱 파일은 데이터가 없어 접속 키 없이 제공한다.\\n                self.reply(200, (ROOT/name).read_bytes(), PUBLIC_FILES[name])\\n                return\\n            if not self.authorized():\\n                self.reply(401, {\'error\': \'코랩 출력에 표시된 전용 접속 링크로 다시 열어주세요.\'})\\n                return\\n            if url.path in (\'/\', \'/dashboard\', \'/dashboard/\'):\\n                self.reply(200, (ROOT/\'index.html\').read_bytes(), \'text/html; charset=utf-8\')\\n            elif url.path in (\'/app.js\', \'/dashboard/app.js\'):\\n                self.reply(200, (ROOT/\'app.js\').read_bytes(), \'text/javascript; charset=utf-8\')\\n            elif (m := API.search(url.path)):\\n                try:\\n                    if not m.group(1):\\n                        self.reply(200, projects.list())\\n                    elif m.group(2) == \'state\':\\n                        self.reply(200, projects.get(m.group(1)).read())\\n                    else:\\n                        self.reply(200, projects.get(m.group(1)).read(), headers={\\n                            \'Content-Disposition\': \'attachment; filename=\\"project-backup.json\\"\'})\\n                except LookupError as exc:\\n                    self.reply(404, {\'error\': str(exc)})\\n            else:\\n                self.reply(404, {\'error\': \'찾을 수 없습니다.\'})\\n\\n        def do_POST(self):\\n            if not self.authorized():\\n                self.reply(401, {\'error\': \'접속 인증이 만료되었습니다.\'}); return\\n            m = API.search(urlsplit(self.path).path)\\n            if not m or m.group(2) == \'export\':\\n                self.reply(404, {\'error\': \'찾을 수 없습니다.\'}); return\\n            if self.headers.get(\'Content-Type\',\'\').split(\';\')[0].strip() != \'application/json\':\\n                self.reply(415, {\'error\': \'JSON 형식만 지원합니다.\'}); return\\n            try:\\n                length = int(self.headers.get(\'Content-Length\', \'0\'))\\n                if length <= 0 or length > 2_000_000:\\n                    raise ValueError(\'저장할 데이터는 2MB 이하여야 합니다.\')\\n                body = json.loads(self.rfile.read(length))\\n                if not isinstance(body, dict):\\n                    raise ValueError(\'요청 형식이 올바르지 않습니다.\')\\n                if m.group(1):\\n                    self.reply(200, projects.get(m.group(1)).save(body[\'state\'], body[\'revision\']))\\n                else:\\n                    title = body.get(\'title\', \'\')\\n                    if not isinstance(title, str) or len(title) > 200:\\n                        raise ValueError(\'프로젝트 제목은 200자 이하여야 합니다.\')\\n                    self.reply(200, projects.create(title.strip()))\\n            except LookupError as exc:\\n                self.reply(404, {\'error\': str(exc)})\\n            except RuntimeError as exc:\\n                self.reply(409, {\'error\': str(exc)})\\n            except (ValueError, KeyError, TypeError) as exc:\\n                self.reply(400, {\'error\': str(exc)})\\n            except OSError:\\n                self.reply(500, {\'error\': \'저장 공간에 기록하지 못했습니다. 백업을 내려받고 다시 시도하세요.\'})\\n    server = ThreadingHTTPServer((\'127.0.0.1\', port), Handler)\\n    thread = threading.Thread(target=server.serve_forever, daemon=True)\\n    thread.start()\\n    server.access_token = token\\n    server.projects = projects\\n    return server\\n\\n\\nif __name__ == \'__main__\':\\n    import argparse\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\'--port\', type=int, default=8765)\\n    args = parser.parse_args()\\n    server = start_server(args.port)\\n    print(f\'대시보드: http://127.0.0.1:{server.server_port}/?key={server.access_token}\', flush=True)\\n    print(\'종료: Ctrl+C. 데이터: data/private/projects/\', flush=True)\\n    try:\\n        threading.Event().wait()\\n    except KeyboardInterrupt:\\n        server.shutdown()\\n", "dashboard/colab.py": "\\"\\"\\"코랩의 기본 전용 링크와 선택적인 공개 공유 링크.\\"\\"\\"\\nimport html\\nimport urllib.error\\nimport urllib.request\\n\\n\\ndef private_link(server):\\n    from google.colab import output\\n    base = output.eval_js(f\'google.colab.kernel.proxyPort({server.server_port})\')\\n    return base.rstrip(\'/\') + \'/?key=\' + server.access_token\\n\\n\\ndef public_link(server):\\n    \\"\\"\\"Gradio 공유 터널로 같은 대시보드 서버를 연결한다.\\n\\n    패키지 설치는 호출자가 Colab에서만 수행한다. 터널은 런타임과 함께 종료된다.\\n    \\"\\"\\"\\n    import gradio as gr\\n    from fastapi import Request\\n    from starlette.responses import Response\\n    with gr.Blocks(analytics_enabled=False) as demo:\\n        gr.Markdown(\'PM System 대시보드 · 코랩 출력의 전용 접속 링크를 사용하세요.\')\\n    demo.launch(share=True, inline=False, prevent_thread_lock=True, quiet=True,\\n                show_api=False)\\n\\n    async def proxy(request: Request, path: str = \'\'):\\n        # 동기 파일 저장/요청은 이벤트 루프 바깥에서 수행한다.\\n        import asyncio\\n        body = await request.body()\\n        if len(body) > 2_000_000:\\n            return Response(\'요청 크기 초과\', status_code=413)\\n        def forward():\\n            class NoRedirect(urllib.request.HTTPRedirectHandler):\\n                def redirect_request(self, *_):\\n                    return None\\n            query = request.url.query\\n            target = f\'http://127.0.0.1:{server.server_port}/\' + path\\n            if query:\\n                target += \'?\' + query\\n            headers = {k: request.headers[k] for k in (\'cookie\',\'content-type\') if k in request.headers}\\n            req = urllib.request.Request(target, data=body if request.method == \'POST\' else None,\\n                                         method=request.method, headers=headers)\\n            try:\\n                response = urllib.request.build_opener(NoRedirect).open(req, timeout=20)\\n            except urllib.error.HTTPError as exc:\\n                response = exc\\n            with response:\\n                data = response.read()\\n                allowed = (\'content-type\',\'set-cookie\',\'location\',\'content-disposition\',\'cache-control\')\\n                out = {k: v for k, v in response.headers.items() if k.lower() in allowed}\\n                return Response(data, status_code=response.code, headers=out)\\n        return await asyncio.to_thread(forward)\\n\\n    demo.app.add_api_route(\'/dashboard/\', proxy, methods=[\'GET\',\'POST\'])\\n    demo.app.add_api_route(\'/dashboard/{path:path}\', proxy, methods=[\'GET\',\'POST\'])\\n    if not demo.share_url:\\n        demo.close()\\n        raise RuntimeError(\'공개 터널을 만들지 못했습니다. 코랩 전용 링크를 사용하세요.\')\\n    return demo, demo.share_url.rstrip(\'/\') + \'/dashboard/?key=\' + server.access_token\\n\\n\\ndef show_link(url, label):\\n    from IPython.display import HTML, display\\n    display(HTML(f\'<p><a target=\\"_blank\\" rel=\\"noopener\\" href=\\"{html.escape(url, quote=True)}\\" \'\\n                 \'style=\\"display:inline-block;padding:14px 22px;background:#1b303d;color:white;\'\\n                 f\'border-radius:9px;text-decoration:none\\">{html.escape(label)}</a></p>\'))\\n", "dashboard/index.html": "<!doctype html>\\n<html lang=\\"ko\\"><head><meta charset=\\"utf-8\\"><meta name=\\"viewport\\" content=\\"width=device-width,initial-scale=1\\"><meta name=\\"theme-color\\" content=\\"#1b303d\\"><link rel=\\"manifest\\" href=\\"manifest.webmanifest\\"><link rel=\\"icon\\" href=\\"icon.svg\\" type=\\"image/svg+xml\\"><link rel=\\"apple-touch-icon\\" href=\\"icon.svg\\"><title>PM System · 의사결정 보조</title>\\n<style>\\n:root{--ink:#1b303d;--muted:#617584;--line:#dce5eb;--blue:#366af0;--teal:#218a7a;--amber:#a76313;--red:#c34840}*{box-sizing:border-box}body{margin:0;background:#f5f7fa;color:var(--ink);font:15px/1.6 system-ui,\'Malgun Gothic\',sans-serif}button,input,select,textarea{font:inherit}button{cursor:pointer;border:1px solid var(--line);border-radius:8px;background:white;padding:8px 14px;color:var(--ink)}button:hover{background:#edf2f7}button:focus-visible,input:focus-visible,textarea:focus-visible,select:focus-visible{outline:3px solid #91afff;outline-offset:2px}.primary{background:var(--ink);color:white;border-color:var(--ink)}.primary:hover{background:#345364}.danger{color:var(--red)}header{height:64px;background:white;border-bottom:1px solid var(--line);display:flex;align-items:center;justify-content:space-between;padding:0 28px;gap:16px}.brand{font-weight:800;letter-spacing:2px;color:var(--teal)}.actions,.row{display:flex;gap:9px;align-items:center;flex-wrap:wrap}.shell{padding:24px 28px 0}.heading{display:flex;justify-content:space-between;gap:14px;align-items:center;margin-bottom:20px}h1{font-size:27px;margin:0}h2{font-size:17px;margin:0 0 8px}p{margin:5px 0}.muted{color:var(--muted);font-size:13px}.pinned{display:grid;grid-template-columns:minmax(0,1.45fr) minmax(300px,1fr);gap:18px}.panel{border:1px solid var(--line);border-radius:13px;background:white;padding:18px 22px;min-width:0}.eyebrow{font-size:11px;letter-spacing:1.6px;color:var(--teal);font-weight:700}.purpose{font-size:19px;font-weight:650;white-space:pre-wrap;overflow-wrap:anywhere;margin:5px 0 8px}.goals{display:flex;gap:8px;flex-wrap:wrap;margin-top:10px}.goal{padding:4px 10px;background:#f1f5f8;font-size:13px;max-width:100%;overflow-wrap:anywhere}.attention{display:grid;grid-template-columns:1fr 1fr;gap:10px}.number{font-size:26px;font-weight:700;color:var(--amber)}.attention button{text-align:left}.notice{border:0;padding:3px 0;background:none;text-align:left;color:var(--blue);display:block;max-width:100%;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.toolbar{padding:20px 0 14px;display:flex;justify-content:space-between;gap:12px;align-items:center;flex-wrap:wrap}.legend{font-size:12px;color:var(--muted)}.legend span{margin-right:10px}.viewport{height:570px;overflow:auto;position:relative;touch-action:pan-x pan-y;border-top:1px solid var(--line);border-bottom:1px solid var(--line);background:#fff;cursor:grab}.viewport.dragging{cursor:grabbing;user-select:none}.world{position:relative;min-height:570px}.ruler{position:sticky;top:0;z-index:5;height:46px;background:#fffffff5;border-bottom:1px solid #e9eef3}.tick{position:absolute;top:0;height:46px;border-left:1px solid #dce5eb;padding:10px 7px;font-size:12px;color:var(--muted);white-space:nowrap}.grid{position:absolute;top:46px;bottom:0;border-left:1px solid #eef2f5;pointer-events:none}.grid.major{border-color:#dce5eb}.axis{position:absolute;height:3px;background:#d4dfe5;left:24px;right:24px}.stem{position:absolute;width:2px;opacity:.55}.task{position:absolute;width:230px;text-align:left;padding:12px 15px;border:1px solid var(--line);border-left:3px solid var(--task-color);border-radius:10px;background:white;box-shadow:0 3px 9px #1b303d06;cursor:pointer;z-index:2}.task b{font-size:15px;display:block;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.task .status{font-size:11px;font-weight:700;color:var(--task-color)}.task .meta{font-size:12px;color:var(--muted);display:block}.task .note{font-size:12px;color:var(--muted);display:block;white-space:nowrap;text-overflow:ellipsis;overflow:hidden}.duration{position:absolute;height:4px;border-radius:2px}.milestone{position:absolute;border:2px solid var(--blue);background:white;width:13px;height:13px;border-radius:50%;z-index:3}.milestone-label{position:absolute;font-size:11px;color:var(--muted);white-space:nowrap}.today{position:absolute;width:2px;background:var(--blue);top:46px;bottom:14px;pointer-events:none;z-index:1}.today-label{position:absolute;background:var(--blue);color:white;font-size:11px;padding:3px 7px;border-radius:5px;top:4px;z-index:6;white-space:nowrap}.empty{position:absolute;top:100px;left:50px;max-width:420px;padding:20px;color:var(--muted)}footer{padding:12px 28px;display:flex;justify-content:space-between;font-size:12px;color:var(--muted)}dialog{border:1px solid var(--line);border-radius:15px;width:min(600px,94vw);max-height:90vh;padding:24px;color:var(--ink)}dialog::backdrop{background:#14273470}form{display:grid;gap:12px}label{display:grid;gap:4px;font-size:13px}input,textarea,select{border:1px solid #cbd7df;border-radius:7px;padding:9px 11px;width:100%;background:white;color:var(--ink)}textarea{resize:vertical;min-height:74px}.cols{display:grid;grid-template-columns:1fr 1fr;gap:12px}.form-actions{display:flex;justify-content:space-between;gap:10px;margin-top:8px}.error{color:var(--red);white-space:pre-wrap}.list-item{border-bottom:1px solid var(--line);padding:10px 0;display:flex;justify-content:space-between;gap:10px}.list-item button{max-width:100%;overflow-wrap:anywhere;text-align:left}.toast{position:fixed;bottom:24px;left:50%;transform:translateX(-50%);padding:10px 18px;background:var(--ink);color:white;border-radius:9px;z-index:20;max-width:90vw}.hidden{display:none!important}.detail-info{background:#f3f6fa;padding:8px 12px;border-radius:6px;font-size:12px}.tabs{display:flex;gap:4px;align-items:center}.tab{border:0;background:none;padding:8px 12px;color:var(--muted);border-radius:8px;font-weight:600}.tab[aria-current=page]{background:#e8f2f0;color:var(--teal)}.chat-link{font-size:13px;color:var(--muted);padding:6px 10px;border:1px dashed var(--line);border-radius:8px;text-decoration:none}a.chat-link{color:var(--blue);border-style:solid}.view{min-height:calc(100vh - 64px)}.back{padding:6px 10px;font-size:13px}.crumb{font-weight:700;max-width:220px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.crumb:empty{display:none}.me-row{display:flex;gap:12px;align-items:flex-end;flex-wrap:wrap;margin-top:16px}.me-row label{width:260px}.project-card .purpose-line{display:-webkit-box;-webkit-line-clamp:2;-webkit-box-orient:vertical;overflow:hidden;white-space:pre-wrap}button.brand{border:0;background:none;padding:0 4px;font:inherit;font-weight:800;letter-spacing:2px;color:var(--teal)}.home{max-width:960px;margin:0 auto;padding:28px 28px 60px}.home h1{margin-bottom:4px}.entry-grid{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin:22px 0 16px}.entry{display:flex;flex-direction:column;gap:6px;text-align:left;padding:22px 24px;border:1px solid var(--line);border-radius:14px;background:white;min-width:0}.entry b{font-size:20px}button.entry:hover{border-color:var(--teal);background:#fbfdfd}.entry-stats{display:flex;gap:14px;flex-wrap:wrap;margin-top:8px;font-size:13px;color:var(--muted)}.entry-stats strong{font-size:20px;color:var(--ink);margin-right:3px}.chat-entry{background:#f8f9fc;border-style:dashed;margin-bottom:16px}.chat-entry b{font-size:16px}.home-todo{margin-top:4px}.todo-item{display:flex;justify-content:space-between;gap:10px;width:100%;text-align:left;border:0;border-bottom:1px solid var(--line);border-radius:0;padding:10px 2px;background:none}.todo-item .muted{white-space:nowrap}.mission{max-width:860px;margin:0 auto;padding:24px 28px 60px}.mission-head{display:flex;justify-content:space-between;gap:12px;align-items:flex-end;flex-wrap:wrap;margin-bottom:16px}.mission-head label{min-width:200px}.counts{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:18px}.counts .panel{padding:12px 14px}.counts .number{font-size:22px}.group{margin-bottom:20px}.group h2{font-size:15px;color:var(--muted)}.mission-item{display:block;width:100%;text-align:left;margin-bottom:8px;padding:12px 14px;border-left:3px solid var(--task-color);border-radius:10px;background:white}.mission-item b{display:block;overflow-wrap:anywhere}.mission-item .meta{font-size:12px;color:var(--muted)}@media(max-width:760px){.home{padding:18px 14px 40px}.entry-grid{grid-template-columns:1fr}.entry{padding:18px}.tabs{position:fixed;left:0;right:0;bottom:0;z-index:15;background:white;border-top:1px solid var(--line);justify-content:space-around;padding:6px 6px calc(6px + env(safe-area-inset-bottom))}.tab{flex:1}body{padding-bottom:64px}.mission{padding:18px 14px 40px}.counts{grid-template-columns:1fr 1fr}header{height:auto;min-height:64px;padding:12px 16px;flex-wrap:wrap}.shell{padding:18px 14px 0}.pinned{grid-template-columns:1fr}.panel{padding:15px}.heading{align-items:flex-start}h1{font-size:22px}.viewport{height:520px}.toolbar .row{gap:5px}.toolbar button{padding:6px 10px}.cols{grid-template-columns:1fr}footer{padding:12px 14px;flex-wrap:wrap}.legend{display:none}}\\n</style></head><body>\\n<header><div class=\\"row\\"><button id=\\"brand\\" class=\\"brand\\" aria-label=\\"내 프로젝트 목록으로\\">PM SYSTEM</button><button id=\\"backToProjects\\" class=\\"back hidden\\">← 내 프로젝트</button><span id=\\"crumb\\" class=\\"crumb\\"></span><nav id=\\"projectTabs\\" class=\\"tabs\\" aria-label=\\"프로젝트 화면\\"><button class=\\"tab\\" data-view=\\"home\\" aria-current=\\"page\\">처음</button><button class=\\"tab\\" data-view=\\"dashboard\\">대시보드</button><button class=\\"tab\\" data-view=\\"missions\\">내 임무</button></nav><span id=\\"chatLink\\" class=\\"chat-link\\" title=\\"팀 채팅은 Discord에서 합니다. 연결 설정은 이후 단계에서 추가됩니다.\\">팀 채팅 · Discord (연결 전)</span></div><nav class=\\"actions\\" id=\\"dashboardActions\\"><button id=\\"reload\\">새로고침</button><button id=\\"export\\">백업 저장</button><button id=\\"import\\">백업 불러오기</button><input type=\\"file\\" id=\\"file\\" accept=\\"application/json,.json\\" hidden></nav></header>\\n<section id=\\"projectsView\\" class=\\"view\\" aria-label=\\"내 프로젝트 목록\\"><div class=\\"home\\"><div class=\\"eyebrow\\">MY PROJECTS</div><h1>내 프로젝트</h1><p class=\\"muted\\">로그인 기능이 생기기 전까지는 내 이름을 직접 입력합니다.</p>\\n<div class=\\"me-row\\"><label>내 이름<input id=\\"me\\" list=\\"meOptions\\" autocomplete=\\"off\\" placeholder=\\"작업 담당자 이름\\"></label><datalist id=\\"meOptions\\"></datalist><button id=\\"newProject\\" class=\\"primary\\">새 프로젝트</button></div>\\n<p id=\\"projectsNote\\" class=\\"muted\\"></p><div id=\\"projectList\\" class=\\"entry-grid\\"></div></div></section>\\n<section id=\\"homeView\\" class=\\"view hidden\\" aria-label=\\"처음 시작 창\\"><div class=\\"home\\"><div class=\\"eyebrow\\">START</div><h1 id=\\"homeTitle\\">불러오는 중…</h1><p id=\\"homeMeta\\" class=\\"muted\\"></p>\\n<div class=\\"entry-grid\\"><button id=\\"goDashboard\\" class=\\"entry\\"><span class=\\"eyebrow\\">대시보드 창</span><b>프로젝트 전체 흐름</b><span class=\\"muted\\">목표·작업 타임라인과 결정 대기</span><span class=\\"entry-stats\\" id=\\"dashStats\\"></span></button>\\n<button id=\\"goMissions\\" class=\\"entry\\"><span class=\\"eyebrow\\">개인별 임무 창</span><b id=\\"missionEntryTitle\\">내 임무</b><span class=\\"muted\\">나에게 배정된 작업과 기한</span><span class=\\"entry-stats\\" id=\\"missionStats\\"></span></button></div>\\n<div class=\\"entry chat-entry\\"><span class=\\"eyebrow\\">팀 채팅 · Discord</span><b>팀 대화는 Discord에서 합니다</b><span class=\\"muted\\">대화에서 찾은 새 목표·작업은 이 창에 알림으로 옵니다. 추가할지는 사람이 정합니다.</span><span class=\\"entry-stats\\"><span id=\\"chatStatus\\">연결 전</span></span></div>\\n<section class=\\"panel home-todo\\"><h2>지금 확인할 일</h2><div id=\\"homeTodo\\"></div></section></div></section>\\n<div id=\\"dashboardView\\" class=\\"view hidden\\"><main class=\\"shell\\"><section class=\\"heading\\"><div><div class=\\"eyebrow\\">PROJECT TIMELINE</div><h1 id=\\"projectTitle\\">불러오는 중…</h1><div id=\\"saved\\" class=\\"muted\\" role=\\"status\\"></div></div><button id=\\"editProject\\">프로젝트 편집</button></section>\\n<section class=\\"pinned\\" aria-label=\\"확대·이동과 무관한 핵심 정보\\"><article class=\\"panel\\"><div class=\\"eyebrow\\">목표와 성공 조건</div><div id=\\"purpose\\" class=\\"purpose\\"></div><p id=\\"success\\" class=\\"muted\\"></p><p id=\\"period\\" class=\\"muted\\"></p><div class=\\"row\\"><h2 style=\\"margin:10px 0 0\\">세부 목표</h2><button id=\\"addGoal\\">목표 추가</button></div><div id=\\"goals\\" class=\\"goals\\"></div></article><article class=\\"panel\\"><h2>지금 확인할 정보</h2><div class=\\"attention\\"><button id=\\"openDecisions\\"><span class=\\"muted\\">결정 대기</span><div id=\\"decisionCount\\" class=\\"number\\">0</div></button><button id=\\"showIssues\\"><span class=\\"muted\\">지연·차단 작업</span><div id=\\"issueCount\\" class=\\"number\\">0</div></button></div><div id=\\"alerts\\"></div><button id=\\"addDecision\\" class=\\"notice\\">결정 요청 추가</button></article></section>\\n<section class=\\"toolbar\\"><div><strong id=\\"range\\"></strong><div class=\\"legend\\"><span>청록 · 완료</span><span>파랑 · 진행 중</span><span>회색 · 예정</span><span>주황 · 차단</span></div></div><div class=\\"row\\"><button id=\\"today\\">오늘</button><button id=\\"fit\\">전체 맞춤</button><button id=\\"zoomOut\\" aria-label=\\"시간축 축소\\">−</button><output id=\\"zoom\\">100%</output><button id=\\"zoomIn\\" aria-label=\\"시간축 확대\\">+</button><button id=\\"addTask\\" class=\\"primary\\">작업 추가</button></div></section></main>\\n<section id=\\"viewport\\" class=\\"viewport\\" aria-label=\\"프로젝트 타임라인. 가로 스크롤로 이동하고 확대·축소 버튼으로 날짜 간격을 조절하세요.\\" tabindex=\\"0\\"><div id=\\"world\\" class=\\"world\\"><div id=\\"ruler\\" class=\\"ruler\\"></div></div></section>\\n<footer><span>빈 공간 드래그 · 가로 스크롤로 이동 / Ctrl + 휠로 확대·축소 / 작업을 눌러 편집</span><span>서버 파일에 저장 · 코랩 종료 전 백업 저장</span></footer></div>\\n<section id=\\"missionView\\" class=\\"view hidden\\" aria-label=\\"개인별 임무 창\\"><div class=\\"mission\\"><div class=\\"mission-head\\"><div><div class=\\"eyebrow\\">MY MISSIONS</div><h1>개인별 임무</h1><p class=\\"muted\\">담당자로 지정된 작업을 모아 봅니다. 로그인 기능이 생기기 전까지는 이름을 직접 고릅니다.</p></div><label>담당자<select id=\\"member\\"></select></label></div><div class=\\"counts\\"><div class=\\"panel\\"><span class=\\"muted\\">지연·차단</span><div id=\\"mIssue\\" class=\\"number\\">0</div></div><div class=\\"panel\\"><span class=\\"muted\\">진행 중</span><div id=\\"mDoing\\" class=\\"number\\">0</div></div><div class=\\"panel\\"><span class=\\"muted\\">예정</span><div id=\\"mTodo\\" class=\\"number\\">0</div></div><div class=\\"panel\\"><span class=\\"muted\\">완료</span><div id=\\"mDone\\" class=\\"number\\">0</div></div></div><div id=\\"missionList\\"></div></div></section>\\n<dialog id=\\"editor\\"><form id=\\"form\\"><h2 id=\\"formTitle\\"></h2><div id=\\"fields\\"></div><div id=\\"error\\" class=\\"error\\" role=\\"alert\\"></div><div class=\\"form-actions\\"><button id=\\"remove\\" class=\\"danger\\" type=\\"button\\">삭제</button><div class=\\"row\\"><button id=\\"cancel\\" type=\\"button\\">취소</button><button class=\\"primary\\" type=\\"submit\\">저장</button></div></div></form></dialog>\\n<dialog id=\\"newProjectDialog\\"><form id=\\"newProjectForm\\"><h2>새 프로젝트</h2><label>프로젝트 이름<input id=\\"newProjectTitle\\" name=\\"title\\" required maxlength=\\"200\\" autocomplete=\\"off\\" placeholder=\\"예: 신제품 베타 준비\\"></label><div id=\\"newProjectError\\" class=\\"error\\" role=\\"alert\\"></div><div class=\\"form-actions\\"><span></span><div class=\\"row\\"><button id=\\"newProjectCancel\\" type=\\"button\\">취소</button><button class=\\"primary\\" type=\\"submit\\">만들기</button></div></div></form></dialog>\\n<dialog id=\\"listing\\"><h2 id=\\"listTitle\\"></h2><div id=\\"listItems\\"></div><button id=\\"closeList\\" style=\\"margin-top:18px\\">닫기</button></dialog><div id=\\"toast\\" class=\\"toast hidden\\" role=\\"status\\"></div>\\n<script src=app.js></script></body></html>\\n", "dashboard/app.js": "\\n\'use strict\';\\nlet state,editing=null,busy=false,scale=46,origin,endDay,toastTimer;\\nconst $=id=>document.getElementById(id), viewport=$(\'viewport\'),world=$(\'world\');\\nconst DAY=86400000, day=s=>Date.parse(s+\'T00:00:00Z\')/DAY, iso=n=>new Date(n*DAY).toISOString().slice(0,10);\\nconst today=()=>{const d=new Date();return day(d.getFullYear()+\'-\'+String(d.getMonth()+1).padStart(2,\'0\')+\'-\'+String(d.getDate()).padStart(2,\'0\'))};\\nconst colors={\'완료\':\'#218a7a\',\'진행 중\':\'#366af0\',\'예정\':\'#71828e\',\'차단\':\'#a76313\'};\\nconst localMode=location.protocol===\'file:\';\\nconst storageKey=\'pm-system-dashboard-v1:\'+location.pathname;\\n// 이름 변경 전 브라우저에 저장된 데이터를 한 번 옮겨 온다.\\nconst legacyStorageKey=\'roh\'+\'ang-dashboard-v1:\'+location.pathname;\\nconst api=new URL(\'api/\',location.href.endsWith(\'/\')?location.href:location.href+\'/\');\\nfunction initialState(){return {version:1,revision:0,project:{title:\'새 프로젝트\',purpose:\'팀이 달성하려는 목적을 작성하세요.\',success:\'완료 여부를 확인할 기준을 작성하세요.\',start:iso(today()-7),end:iso(today()+21)},goals:[],tasks:[],decisions:[]}}\\nfunction validateState(s){\\nif(!s||s.version!==1||!s.project)throw new Error(\'지원하는 대시보드 백업 파일(version 1)이 아닙니다.\');\\nconst text=(v,max=3000,required=false)=>{if(typeof v!==\'string\'||v.length>max||required&&!v.trim())throw new Error(\'제목과 입력 내용의 형식을 확인하세요.\')};\\nconst dateValue=v=>{if(typeof v!==\'string\'||!/^\\\\d{4}-\\\\d{2}-\\\\d{2}$/.test(v)||!Number.isFinite(day(v))||iso(day(v))!==v)throw new Error(\'올바른 날짜를 입력하세요.\');return day(v)};\\nconst dates=o=>{const a=dateValue(o.start),b=dateValue(o.end);if(a>b||b-a>3650)throw new Error(\'종료일은 시작일 이후이며 기간은 10년 이내여야 합니다.\')};\\nfor(const k of [\'title\',\'purpose\',\'success\'])text(s.project[k],3000,k===\'title\');dates(s.project);\\nconst ids=new Set();for(const kind of [\'goals\',\'tasks\',\'decisions\']){if(!Array.isArray(s[kind])||s[kind].length>500)throw new Error(\'항목은 종류별 500개까지 저장할 수 있습니다.\');for(const item of s[kind]){if(!item||typeof item!==\'object\')throw new Error(\'항목 형식이 올바르지 않습니다.\');text(item.id,100,true);text(item.title,200,true);text(item.note??\'\');if(ids.has(item.id))throw new Error(\'중복된 항목 ID입니다.\');ids.add(item.id);if(kind===\'tasks\'){dates(item);text(item.owner??\'\',100);if(!Object.hasOwn(colors,item.status))throw new Error(\'작업 상태가 올바르지 않습니다.\')}else{dateValue(item.due);if(typeof item.done!==\'boolean\')throw new Error(\'완료 상태가 올바르지 않습니다.\')}}}\\nconst tasks=new Map(s.tasks.map(t=>[t.id,t])),goals=new Set(s.goals.map(g=>g.id));\\nconst horizon=[day(s.project.start),day(s.project.end),...s.tasks.flatMap(t=>[day(t.start),day(t.end)])];if(Math.max(...horizon)-Math.min(...horizon)>3650)throw new Error(\'전체 시간 범위는 10년 이내여야 합니다.\');\\nfor(const t of s.tasks){const seen=new Set([t.id]);let p=t.predecessor;while(p){if(!tasks.has(p)||seen.has(p))throw new Error(\'선행 작업이 없거나 순환 관계입니다.\');seen.add(p);p=tasks.get(p).predecessor}if(t.predecessor&&t.start<tasks.get(t.predecessor).end)throw new Error(\'후속 작업 시작일은 선행 작업 종료일 이후여야 합니다.\');if(t.goal&&!goals.has(t.goal))throw new Error(\'연결된 목표가 없습니다.\')}\\nreturn s;\\n}\\n// 파일 직접 실행 모드: 프로젝트 목록과 프로젝트별 데이터를 이 브라우저에 저장한다.\\nconst indexKey=\'pm-system-projects:\'+location.pathname,pkey=id=>\'pm-system-project:\'+id+\':\'+location.pathname;\\nfunction browserStore(){try{const ls=localStorage;ls.getItem(indexKey);return ls}catch(_){throw new Error(\'브라우저 저장소를 사용할 수 없습니다. 코랩 실행 링크 또는 로컬 서버로 열어주세요.\')}}\\nfunction localIds(){const ls=browserStore(),raw=ls.getItem(indexKey);if(raw!==null)return JSON.parse(raw);const old=ls.getItem(storageKey)??ls.getItem(legacyStorageKey),ids=[];if(old){ls.setItem(pkey(\'local-1\'),old);ids.push(\'local-1\')}ls.setItem(indexKey,JSON.stringify(ids));return ids}\\nfunction readProject(id){return validateState(JSON.parse(browserStore().getItem(pkey(id))))}\\nfunction writeProject(id,s){try{browserStore().setItem(pkey(id),JSON.stringify(s))}catch(_){throw new Error(\'브라우저에 저장하지 못했습니다. 기존 데이터는 유지됩니다. 저장 공간과 설정을 확인하세요.\')}}\\nfunction localRequest(path,options){const ids=localIds(),method=options.method||\'GET\',body=options.body?JSON.parse(options.body):null;\\nif(path===\'projects\'){if(method===\'GET\')return ids.map(id=>({id,state:readProject(id)}));if(ids.length>=50)throw new Error(\'프로젝트는 50개까지 만들 수 있습니다.\');const id=\'local-\'+crypto.randomUUID().slice(0,8),next=initialState();if(body&&body.title)next.project.title=body.title;validateState(next);next.revision=1;writeProject(id,next);browserStore().setItem(indexKey,JSON.stringify([...ids,id]));return {id,state:next}}\\nconst m=/^projects\\\\/([^/]+)\\\\/state$/.exec(path);if(!m||!ids.includes(decodeURIComponent(m[1])))throw new Error(\'프로젝트를 찾을 수 없습니다. 목록에서 다시 선택하세요.\');const id=decodeURIComponent(m[1]),current=readProject(id);if(method===\'GET\')return current;\\nif(body.revision!==current.revision)throw new Error(\'다른 창에서 내용이 바뀌었습니다. 새로고침 후 다시 수정하세요.\');const next=validateState(body.state);next.revision=current.revision+1;writeProject(id,next);return structuredClone(next)}\\nfunction el(tag,cls,txt){const n=document.createElement(tag);if(cls)n.className=cls;if(txt!==undefined)n.textContent=txt;return n}\\nfunction toast(s){$(\'toast\').textContent=s;$(\'toast\').classList.remove(\'hidden\');clearTimeout(toastTimer);toastTimer=setTimeout(()=>$(\'toast\').classList.add(\'hidden\'),4500)}\\nasync function request(path,options={}){if(localMode)return localRequest(path,options);let r;try{r=await fetch(new URL(path,api),options)}catch(_){throw new Error(\'실행 서버에 연결하지 못했습니다. 코랩 런타임과 대시보드 실행 셀을 확인하세요.\')}let b;try{b=await r.json()}catch(_){throw new Error(\'대시보드 연결이 만료되었거나 올바른 실행 링크가 아닙니다. 코랩에서 새 링크를 생성하세요.\')}if(!r.ok)throw new Error(b.error||\'요청에 실패했습니다.\');return b}\\nconst statePath=()=>\'projects/\'+encodeURIComponent(projectId)+\'/state\';\\nasync function load(){if(!projectId)return refreshProjects();try{state=await request(statePath());render();$(\'saved\').textContent=localMode?\'파일 직접 실행 · 이 브라우저에 저장됩니다. 다른 기기와 공유되지 않으니 백업을 저장하세요.\':\'저장된 상태를 불러왔습니다.\'}catch(e){toast(e.message);$(\'saved\').textContent=e.message}}\\nasync function save(next){if(busy)throw new Error(\'저장 중입니다. 잠시 후 다시 시도하세요.\');validateState(next);busy=true;try{state=await request(statePath(),{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({state:next,revision:state.revision})});render();$(\'saved\').textContent=(localMode?\'브라우저 저장 완료 · \':\'서버 저장 완료 · \')+new Date().toLocaleTimeString(\'ko-KR\');return true}finally{busy=false}}\\nfunction issues(){return state.tasks.filter(t=>t.status===\'차단\'||(t.status!==\'완료\'&&day(t.end)<today()))}\\nfunction render(){const p=state.project;$(\'projectTitle\').textContent=p.title;$(\'purpose\').textContent=p.purpose;$(\'success\').textContent=\'성공 조건 · \'+(p.success||\'미작성\');$(\'period\').textContent=p.start+\' — \'+p.end;$(\'goals\').replaceChildren();for(const g of state.goals){const b=el(\'button\',\'goal\',(g.done?\'✓ \':\'\')+g.title);b.onclick=()=>openEditor(\'goals\',g.id);$(\'goals\').append(b)}if(!state.goals.length)$(\'goals\').append(el(\'span\',\'muted\',\'목표를 추가해 작업과 연결하세요.\'));\\nconst pending=state.decisions.filter(d=>!d.done);$(\'decisionCount\').textContent=pending.length;$(\'issueCount\').textContent=issues().length;$(\'alerts\').replaceChildren();for(const d of pending.slice(0,2)){const b=el(\'button\',\'notice\',d.due+\' · \'+d.title);b.onclick=()=>openEditor(\'decisions\',d.id);$(\'alerts\').append(b)}draw();renderMissions();renderHome();$(\'crumb\').textContent=p.title}\\nfunction draw(){if(!state)return;const dates=[day(state.project.start),day(state.project.end),...state.tasks.flatMap(t=>[day(t.start),day(t.end)])];origin=Math.min(...dates)-2;endDay=Math.max(...dates)+3;const width=Math.max(viewport.clientWidth,(endDay-origin)*scale+100);world.style.width=width+\'px\';world.replaceChildren();const ruler=el(\'div\',\'ruler\');world.append(ruler);$(\'range\').textContent=iso(origin)+\' — \'+iso(endDay);$(\'zoom\').textContent=Math.round(scale/46*100)+\'%\';\\nconst tasks=[...state.tasks].sort((a,b)=>a.start.localeCompare(b.start)||a.id.localeCompare(b.id));const lanes=[[],[]],layout=[];tasks.forEach((t,i)=>{const side=i%2,x=50+(day(t.start)-origin)*scale,right=x+Math.max(240,(day(t.end)-day(t.start))*scale);let lane=lanes[side].findIndex(e=>e+18<=x);if(lane<0)lane=lanes[side].length;lanes[side][lane]=right;layout.push({t,side,lane,x})});const upper=Math.max(1,lanes[0].length),axis=90+upper*150,height=axis+Math.max(1,lanes[1].length)*150+90;world.style.height=height+\'px\';\\nfor(let d=origin;d<=endDay;d++){const x=50+(d-origin)*scale,major=new Date(d*DAY).getUTCDay()===1;if(scale>=22||major){const g=el(\'div\',\'grid\'+(major?\' major\':\'\'));g.style.left=x+\'px\';world.append(g)}if(scale>=70||scale>=28&&(d-origin)%2===0||scale<28&&major){const tick=el(\'div\',\'tick\',iso(d).slice(5).replace(\'-\',\'.\'));tick.style.left=x+\'px\';ruler.append(tick)}}\\nconst axisEl=el(\'div\',\'axis\');axisEl.style.top=axis+\'px\';world.append(axisEl);\\nfor(const [d,title] of [[day(state.project.start),\'시작\'],[day(state.project.end),\'목표 기한\']]){const x=50+(d-origin)*scale,n=el(\'div\',\'milestone\');n.style.left=x-6+\'px\';n.style.top=axis-5+\'px\';const label=el(\'div\',\'milestone-label\',title);label.style.left=x-12+\'px\';label.style.top=axis+15+\'px\';world.append(n,label)}\\nconst nx=50+(today()-origin)*scale;if(nx>=0&&nx<width){const n=el(\'div\',\'today\');n.style.left=nx+\'px\';const label=el(\'div\',\'today-label\',\'오늘 \'+iso(today()).slice(5));label.style.left=nx-26+\'px\';ruler.append(label);world.append(n)}\\nfor(const {t,side,lane,x} of layout){const y=side?axis+60+lane*150:axis-145-lane*150,c=colors[t.status],stem=el(\'div\',\'stem\');stem.style.cssText=`left:${x}px;top:${Math.min(axis,side?y:y+106)}px;height:${Math.abs(axis-(side?y:y+106))}px;background:${c}`;const b=el(\'button\',\'task\');b.style.cssText=`left:${x}px;top:${y}px;--task-color:${c}`;b.append(el(\'span\',\'status\',t.status),el(\'b\',\'\',t.title),el(\'span\',\'meta\',t.start.slice(5)+\' — \'+t.end.slice(5)+\' · \'+(t.owner||\'담당 미정\')));if(scale>=35)b.append(el(\'span\',\'note\',t.note||\'진행 내용을 작성하세요\'));b.onclick=()=>openEditor(\'tasks\',t.id);const dur=el(\'div\',\'duration\');dur.style.cssText=`left:${x}px;top:${y+119}px;width:${Math.max(4,(day(t.end)-day(t.start))*scale)}px;background:${c}`;world.append(stem,b,dur)}\\nif(!tasks.length){const e=el(\'div\',\'empty\');e.append(el(\'h2\',\'\',\'아직 작업이 없습니다.\'),el(\'p\',\'\',\'목표와 프로젝트 제목을 작성하고 첫 작업을 추가하세요. 작업은 시작 날짜에 맞춰 중심선에 연결됩니다.\'));world.append(e)}}\\nfunction field(name,title,value,type=\'text\',options=[]){const label=el(\'label\',\'\',title);let input;if(type===\'textarea\')input=el(\'textarea\');else if(type===\'select\'){input=el(\'select\');for(const [v,t]of options){const o=el(\'option\',\'\',t);o.value=v;input.append(o)}}else{input=el(\'input\');input.type=type}input.name=name;input.value=value??\'\';if(name===\'title\')input.required=true;if(type===\'date\')input.required=true;if(type===\'checkbox\')input.checked=!!value;label.append(input);return label}\\nfunction openEditor(kind,id){if(!state)return;const item=kind===\'project\'?state.project:state[kind].find(x=>x.id===id)||{};editing={kind,id};$(\'listing\').close();$(\'error\').textContent=\'\';$(\'formTitle\').textContent={project:\'프로젝트 편집\',goals:\'목표\',tasks:\'작업\',decisions:\'결정 요청\'}[kind]+(kind===\'project\'?\'\':id?\' 편집\':\' 추가\');const f=$(\'fields\');f.replaceChildren();f.append(field(\'title\',\'제목\',item.title));if(kind===\'project\'){f.append(field(\'purpose\',\'목적\',item.purpose,\'textarea\'),field(\'success\',\'성공 조건\',item.success,\'textarea\'))}if(kind===\'project\'||kind===\'tasks\'){f.append(field(\'start\',\'시작일\',item.start||iso(today()),\'date\'),field(\'end\',\'종료일\',item.end||iso(today()+3),\'date\'))}else f.append(field(\'due\',\'기한\',item.due||iso(today()+7),\'date\'),field(\'done\',\'완료 / 결정 완료\',item.done,\'checkbox\'));\\nif(kind===\'tasks\'){f.append(field(\'owner\',\'담당자\',item.owner),field(\'status\',\'상태\',item.status||\'예정\',\'select\',Object.keys(colors).map(v=>[v,v])),field(\'predecessor\',\'선행 작업 (완료 후 시작)\',item.predecessor,\'select\',[[\'\',\'없음\'],...state.tasks.filter(t=>t.id!==id).map(t=>[t.id,t.title])]),field(\'goal\',\'연결 목표\',item.goal,\'select\',[[\'\',\'없음\'],...state.goals.map(g=>[g.id,g.title])]));const pred=state.tasks.find(t=>t.id===item.predecessor);if(pred)f.append(el(\'p\',\'detail-info\',\'선행 작업 종료: \'+pred.end+\' · \'+pred.title))}if(kind!==\'project\')f.append(field(\'note\',kind===\'decisions\'?\'배경·선택지·관련 의견\':\'내용·진행 상황\',item.note,\'textarea\'));$(\'remove\').classList.toggle(\'hidden\',!id||kind===\'project\');$(\'editor\').showModal()}\\n$(\'form\').onsubmit=async e=>{e.preventDefault();const next=structuredClone(state),data=Object.fromEntries(new FormData(e.target));for(const name of [\'done\'])if(e.target.elements[name])data[name]=e.target.elements[name].checked;const {kind,id}=editing;if(kind===\'project\')next.project=data;else{data.id=id||crypto.randomUUID();const i=next[kind].findIndex(x=>x.id===id);if(i<0)next[kind].push(data);else next[kind][i]=data}try{await save(next);$(\'editor\').close();toast(\'저장했습니다.\')}catch(err){$(\'error\').textContent=err.message}};\\n$(\'remove\').onclick=async()=>{const {kind,id}=editing;if(!confirm(\'이 항목을 삭제할까요? 연결된 목표·선행 작업 참조도 해제됩니다.\'))return;const next=structuredClone(state);next[kind]=next[kind].filter(x=>x.id!==id);for(const t of next.tasks){if(kind===\'goals\'&&t.goal===id)t.goal=\'\';if(kind===\'tasks\'&&t.predecessor===id)t.predecessor=\'\'}try{await save(next);$(\'editor\').close()}catch(e){$(\'error\').textContent=e.message}};\\nfunction listing(title,items,kind){$(\'listTitle\').textContent=title;$(\'listItems\').replaceChildren();for(const t of items){const row=el(\'div\',\'list-item\');const b=el(\'button\',\'\',t.title+\' · \'+(t.due||t.end));b.onclick=()=>{if(kind===\'tasks\')focusTask(t.id);openEditor(kind,t.id)};row.append(b);$(\'listItems\').append(row)}if(!items.length)$(\'listItems\').append(el(\'p\',\'muted\',\'해당 항목이 없습니다.\'));$(\'listing\').showModal()}\\nfunction focusTask(id){const t=state.tasks.find(t=>t.id===id);if(t)viewport.scrollLeft=Math.max(0,50+(day(t.start)-origin)*scale-120)}\\nfunction zoom(factor,anchor=viewport.clientWidth/2){const dateAt=origin+(viewport.scrollLeft+anchor-50)/scale;scale=Math.max(3,Math.min(220,scale*factor));draw();viewport.scrollLeft=50+(dateAt-origin)*scale-anchor}\\n$(\'zoomIn\').onclick=()=>zoom(1.4);$(\'zoomOut\').onclick=()=>zoom(1/1.4);$(\'fit\').onclick=()=>{if(!state)return;scale=Math.max(3,Math.min(150,(viewport.clientWidth-120)/(endDay-origin)));draw();viewport.scrollLeft=0};$(\'today\').onclick=()=>viewport.scrollLeft=Math.max(0,50+(today()-origin)*scale-viewport.clientWidth/2);\\nviewport.addEventListener(\'wheel\',e=>{if(e.ctrlKey){e.preventDefault();zoom(e.deltaY<0?1.12:1/1.12,e.clientX-viewport.getBoundingClientRect().left)}},{passive:false});let drag=null;viewport.addEventListener(\'pointerdown\',e=>{if(e.button!==0||e.target.closest(\'button\'))return;drag={x:e.clientX,y:e.clientY,left:viewport.scrollLeft,top:viewport.scrollTop};viewport.setPointerCapture(e.pointerId);viewport.classList.add(\'dragging\')});viewport.addEventListener(\'pointermove\',e=>{if(drag){viewport.scrollLeft=drag.left+drag.x-e.clientX;viewport.scrollTop=drag.top+drag.y-e.clientY}});for(const event of [\'pointerup\',\'pointercancel\'])viewport.addEventListener(event,()=>{drag=null;viewport.classList.remove(\'dragging\')});\\n$(\'editProject\').onclick=()=>openEditor(\'project\');$(\'addGoal\').onclick=()=>openEditor(\'goals\');$(\'addTask\').onclick=()=>openEditor(\'tasks\');$(\'addDecision\').onclick=()=>openEditor(\'decisions\');$(\'cancel\').onclick=()=>$(\'editor\').close();$(\'closeList\').onclick=()=>$(\'listing\').close();$(\'openDecisions\').onclick=()=>listing(\'결정 요청\',state.decisions,\'decisions\');$(\'showIssues\').onclick=()=>listing(\'지연·차단 작업\',issues(),\'tasks\');$(\'reload\').onclick=load;\\n$(\'export\').onclick=()=>{if(!state){toast(\'먼저 대시보드를 불러와주세요.\');return}if(!localMode){location.href=new URL(\'projects/\'+encodeURIComponent(projectId)+\'/export\',api).href;return}const url=URL.createObjectURL(new Blob([JSON.stringify(state,null,2)],{type:\'application/json\'})),link=el(\'a\');link.href=url;link.download=\'project-backup.json\';document.body.append(link);link.click();link.remove();setTimeout(()=>URL.revokeObjectURL(url),1000)};$(\'import\').onclick=()=>$(\'file\').click();$(\'file\').onchange=async e=>{const f=e.target.files[0];if(!f)return;try{if(f.size>2_000_000)throw new Error(\'2MB 이하의 백업 파일을 선택하세요.\');const next=JSON.parse(await f.text());if(!confirm(\'현재 내용을 백업 파일의 내용으로 교체할까요? 필요하면 먼저 백업 저장을 누르세요.\'))return;await save(next);toast(\'백업을 불러왔습니다.\')}catch(err){toast(err.message)}finally{e.target.value=\'\'}};\\nif(localMode){document.querySelector(\'footer span:last-child\').textContent=\'브라우저에 저장 · 백업 파일로 코랩에 옮기기\';}\\n// 앱 틀: 내 프로젝트 목록 → 프로젝트별 처음 시작 창 → 하위 창(대시보드·개인별 임무).\\n// 팀 채팅은 앱 밖 Discord에서 한다. 로그인 전까지는 \'내 이름\'을 직접 고른다.\\nlet projects=[],projectId=null;\\nconst VIEWS={projects:\'projectsView\',home:\'homeView\',dashboard:\'dashboardView\',missions:\'missionView\'},meKey=\'pm-system-me:\'+location.pathname;\\nfunction parseHash(){const [a,b]=(location.hash||\'\').replace(\'#\',\'\').split(\'/\');if(!a||a===\'projects\')return {pid:null,view:\'projects\'};return {pid:decodeURIComponent(a),view:b&&b!==\'projects\'&&Object.hasOwn(VIEWS,b)?b:\'home\'}}\\nfunction currentView(){return parseHash().view}\\nfunction showView(v){for(const [name,id] of Object.entries(VIEWS))$(id).classList.toggle(\'hidden\',name!==v);$(\'dashboardActions\').classList.toggle(\'hidden\',v!==\'dashboard\');$(\'projectTabs\').classList.toggle(\'hidden\',v===\'projects\');$(\'backToProjects\').classList.toggle(\'hidden\',v===\'projects\');$(\'crumb\').textContent=v===\'projects\'||!state?\'\':state.project.title;for(const t of document.querySelectorAll(\'.tab\'))t.setAttribute(\'aria-current\',t.dataset.view===v?\'page\':\'false\');if(v===\'projects\')renderProjects();else if(!state)return;else if(v===\'dashboard\')draw();else if(v===\'missions\')renderMissions();else renderHome()}\\nfunction go(v,pid=projectId){location.hash=v===\'projects\'?\'projects\':encodeURIComponent(pid)+\'/\'+v;return route()}\\nasync function route(){const {pid,view}=parseHash();if(!pid){projectId=null;state=undefined;showView(\'projects\');return refreshProjects()}\\nif(pid!==projectId||!state){projectId=pid;state=undefined;chosenMember=null;try{state=await request(statePath());render()}catch(e){toast(e.message);location.hash=\'projects\';projectId=null;showView(\'projects\');return refreshProjects()}}showView(view)}\\nfor(const t of document.querySelectorAll(\'.tab\'))t.onclick=()=>go(t.dataset.view);\\n$(\'brand\').onclick=()=>go(\'projects\');$(\'backToProjects\').onclick=()=>go(\'projects\');$(\'goDashboard\').onclick=()=>go(\'dashboard\');$(\'goMissions\').onclick=()=>go(\'missions\');\\nif(typeof globalThis.addEventListener===\'function\')globalThis.addEventListener(\'hashchange\',()=>route());\\nfunction stat(n,label){const s=el(\'span\');s.append(el(\'strong\',\'\',String(n)),label);return s}\\nfunction me(){try{return (localStorage.getItem(meKey)||\'\').trim()}catch(_){return \'\'}}\\nconst lateTasks=s=>s.tasks.filter(t=>t.status===\'차단\'||(t.status!==\'완료\'&&day(t.end)<today()));\\n// 내 프로젝트 목록: 프로젝트마다 기간, 내 임무, 결정 대기를 보여 주고 누르면 그 프로젝트의 처음 시작 창으로 간다.\\nasync function refreshProjects(){try{projects=await request(\'projects\');renderProjects()}catch(e){toast(e.message);$(\'projectList\').replaceChildren(el(\'p\',\'error\',e.message))}}\\nfunction renderProjects(){const who=me();$(\'me\').value=who;const names=[...new Set(projects.flatMap(p=>p.state.tasks.map(t=>(t.owner||\'\').trim())).filter(Boolean))].sort((a,b)=>a.localeCompare(b,\'ko\'));$(\'meOptions\').replaceChildren(...names.map(n=>{const o=el(\'option\');o.value=n;return o}));\\nconst mine=p=>p.state.tasks.filter(t=>who&&(t.owner||\'\').trim()===who);const sorted=[...projects].sort((a,b)=>(mine(b).length>0)-(mine(a).length>0));$(\'projectList\').replaceChildren();\\nfor(const p of sorted){const s=p.state,my=mine(p),late=new Set(lateTasks(s).map(t=>t.id)),b=el(\'button\',\'entry project-card\');b.append(el(\'span\',\'eyebrow\',s.project.start+\' — \'+s.project.end),el(\'b\',\'\',s.project.title),el(\'span\',\'muted purpose-line\',s.project.purpose));const st=el(\'span\',\'entry-stats\');if(who)st.append(stat(my.filter(t=>t.status!==\'완료\').length,\'내 임무\'),stat(my.filter(t=>late.has(t.id)).length,\'내 지연·차단\'));st.append(stat(s.decisions.filter(d=>!d.done).length,\'결정 대기\'),stat(s.tasks.length,\'전체 작업\'));b.append(st);b.onclick=()=>go(\'home\',p.id);$(\'projectList\').append(b)}\\nif(!projects.length)$(\'projectList\').append(el(\'p\',\'muted\',\'아직 프로젝트가 없습니다. 새 프로젝트를 만들어 시작하세요.\'));$(\'projectsNote\').textContent=who?who+\' 님 기준으로 내 임무를 셉니다. 내 작업이 있는 프로젝트가 먼저 보입니다.\':\'내 이름을 입력하면 프로젝트마다 내 임무 수를 보여 줍니다.\'}\\n$(\'me\').onchange=e=>{try{localStorage.setItem(meKey,e.target.value.trim())}catch(_){}renderProjects()};\\n// 새 프로젝트는 이름부터 받는다. 같은 이름이 이미 있으면 구별되도록 막는다.\\n$(\'newProject\').onclick=()=>{$(\'newProjectTitle\').value=\'\';$(\'newProjectError\').textContent=\'\';$(\'newProjectDialog\').showModal()};\\n$(\'newProjectCancel\').onclick=()=>$(\'newProjectDialog\').close();\\nasync function createProject(title){title=(title||\'\').trim();if(!title)throw new Error(\'프로젝트 이름을 입력하세요.\');if(projects.some(p=>p.state.project.title.trim()===title))throw new Error(\'같은 이름의 프로젝트가 이미 있습니다. 다른 이름을 입력하세요.\');const created=await request(\'projects\',{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify({title})});projects.push(created);await go(\'home\',created.id);return created}\\n$(\'newProjectForm\').onsubmit=async e=>{e.preventDefault();try{await createProject($(\'newProjectTitle\').value);$(\'newProjectDialog\').close();toast(\'프로젝트를 만들었습니다. 목적과 기간은 대시보드의 프로젝트 편집에서 바꿀 수 있습니다.\')}catch(err){$(\'newProjectError\').textContent=err.message}};\\n// 처음 시작 창: 두 하위 창(대시보드·개인별 임무)으로 가는 입구와 지금 확인할 일.\\nfunction renderHome(){if(!state)return;const p=state.project;$(\'homeTitle\').textContent=p.title;$(\'homeMeta\').textContent=p.start+\' — \'+p.end+\' · 오늘 \'+iso(today());\\nconst late=issues(),pending=state.decisions.filter(d=>!d.done);$(\'dashStats\').replaceChildren(stat(state.goals.length,\'목표\'),stat(state.tasks.length,\'작업\'),stat(pending.length,\'결정 대기\'),stat(late.length,\'지연·차단\'));\\nconst list=members(),saved=savedMember(),who=list.includes(saved)?saved:(list[0]??\'\');$(\'missionEntryTitle\').textContent=list.length?(who||\'담당 미정\')+\' 님의 임무\':\'내 임무\';$(\'missionStats\').replaceChildren();if(list.length){for(const [title,,items] of missionGroups(who).slice(0,3))$(\'missionStats\').append(stat(items.length,title))}else $(\'missionStats\').append(el(\'span\',\'\',\'담당자가 지정된 작업이 없습니다\'));\\nconst todo=[...late.map(t=>({kind:\'tasks\',id:t.id,title:t.title,when:t.end,label:t.status===\'차단\'?\'차단\':\'기한 지남\'})),...pending.map(d=>({kind:\'decisions\',id:d.id,title:d.title,when:d.due,label:\'결정 대기\'}))].sort((a,b)=>a.when.localeCompare(b.when)).slice(0,6);\\n$(\'homeTodo\').replaceChildren();for(const t of todo){const b=el(\'button\',\'todo-item\');b.append(el(\'span\',\'\',t.title),el(\'span\',\'muted\',t.label+\' · \'+t.when));b.onclick=()=>openEditor(t.kind,t.id);$(\'homeTodo\').append(b)}if(!todo.length)$(\'homeTodo\').append(el(\'p\',\'muted\',\'지연·차단된 작업이나 결정 대기가 없습니다.\'))}\\nfunction members(){return [...new Set(state.tasks.map(t=>(t.owner||\'\').trim()))].sort((a,b)=>a===\'\'?1:b===\'\'?-1:a.localeCompare(b,\'ko\'))}\\nlet chosenMember=null;\\nfunction savedMember(){return chosenMember!==null?chosenMember:me()}\\nfunction missionGroups(who){const mine=state.tasks.filter(t=>(t.owner||\'\').trim()===who).sort((a,b)=>a.end.localeCompare(b.end)||a.title.localeCompare(b.title,\'ko\'));const late=new Set(issues().map(t=>t.id));return [[\'지연·차단\',\'mIssue\',mine.filter(t=>late.has(t.id))],[\'진행 중\',\'mDoing\',mine.filter(t=>!late.has(t.id)&&t.status===\'진행 중\')],[\'예정\',\'mTodo\',mine.filter(t=>!late.has(t.id)&&t.status===\'예정\')],[\'완료\',\'mDone\',mine.filter(t=>t.status===\'완료\')]]}\\nfunction renderMissions(){if(!state)return;const list=members(),sel=$(\'member\'),saved=savedMember();const who=list.includes(saved)?saved:(list[0]??\'\');sel.replaceChildren();for(const m of list){const o=el(\'option\',\'\',m||\'담당 미정\');o.value=m;sel.append(o)}sel.value=who;$(\'missionList\').replaceChildren();if(!list.length){$(\'missionList\').append(el(\'p\',\'muted\',\'아직 작업이 없습니다. 대시보드에서 작업을 추가하고 담당자를 지정하세요.\'));for(const id of [\'mIssue\',\'mDoing\',\'mTodo\',\'mDone\'])$(id).textContent=\'0\';return}const goals=new Map(state.goals.map(g=>[g.id,g.title]));for(const [title,count,items] of missionGroups(who)){$(count).textContent=items.length;if(!items.length)continue;const g=el(\'section\',\'group\');g.append(el(\'h2\',\'\',title+\' \'+items.length));for(const t of items){const b=el(\'button\',\'mission-item\');b.style.setProperty?.(\'--task-color\',colors[t.status]);b.append(el(\'b\',\'\',t.title),el(\'span\',\'meta\',t.start+\' — \'+t.end+\' · \'+t.status+(t.goal&&goals.has(t.goal)?\' · 목표: \'+goals.get(t.goal):\'\')));if(t.note)b.append(el(\'p\',\'meta\',t.note));b.onclick=()=>openEditor(\'tasks\',t.id);g.append(b)}$(\'missionList\').append(g)}}\\n$(\'member\').onchange=e=>{chosenMember=e.target.value;renderMissions()};\\n// 설치형 앱(PWA). 파일 직접 실행이나 지원하지 않는 브라우저에서는 건너뛴다.\\nif(!localMode&&globalThis.navigator&&\'serviceWorker\' in navigator&&globalThis.isSecureContext)navigator.serviceWorker.register(\'sw.js\').catch(()=>{});\\nroute();\\n", "dashboard/manifest.webmanifest": "{\\n  \\"name\\": \\"PM System 의사결정 보조\\",\\n  \\"short_name\\": \\"PM System\\",\\n  \\"description\\": \\"의사결정과 실행·자원관리를 잇는 범용 의사결정 보조 앱\\",\\n  \\"lang\\": \\"ko\\",\\n  \\"start_url\\": \\"./\\",\\n  \\"scope\\": \\"./\\",\\n  \\"display\\": \\"standalone\\",\\n  \\"background_color\\": \\"#f5f7fa\\",\\n  \\"theme_color\\": \\"#1b303d\\",\\n  \\"icons\\": [\\n    {\\"src\\": \\"icon.svg\\", \\"sizes\\": \\"any\\", \\"type\\": \\"image/svg+xml\\", \\"purpose\\": \\"any\\"},\\n    {\\"src\\": \\"icon.svg\\", \\"sizes\\": \\"any\\", \\"type\\": \\"image/svg+xml\\", \\"purpose\\": \\"maskable\\"}\\n  ]\\n}\\n", "dashboard/sw.js": "\'use strict\';\\n// 앱 화면 파일만 보관한다. 대시보드 데이터(api/)는 보관하지 않고 항상 서버에서 받는다.\\nconst CACHE=\'pm-system-shell-v1\';\\nconst SHELL=[\'./\',\'app.js\',\'manifest.webmanifest\',\'icon.svg\'];\\nself.addEventListener(\'install\',e=>{e.waitUntil(caches.open(CACHE).then(c=>c.addAll(SHELL)).catch(()=>{}));self.skipWaiting()});\\nself.addEventListener(\'activate\',e=>{e.waitUntil(caches.keys().then(keys=>Promise.all(keys.filter(k=>k!==CACHE).map(k=>caches.delete(k)))).then(()=>self.clients.claim()))});\\nself.addEventListener(\'fetch\',e=>{\\n  const url=new URL(e.request.url);\\n  if(e.request.method!==\'GET\'||url.origin!==location.origin||url.pathname.includes(\'/api/\')||url.search.includes(\'key=\'))return;\\n  // 최신 화면을 먼저 받고, 연결이 끊겼을 때만 보관한 화면을 쓴다.\\n  e.respondWith(fetch(e.request).then(r=>{if(r.ok){const copy=r.clone();caches.open(CACHE).then(c=>c.put(e.request,copy))}return r}).catch(()=>caches.match(e.request,{ignoreSearch:true})));\\n});\\n", "dashboard/icon.svg": "<svg xmlns=\\"http://www.w3.org/2000/svg\\" viewBox=\\"0 0 512 512\\"><rect width=\\"512\\" height=\\"512\\" fill=\\"#1b303d\\"/><path d=\\"M96 276h320\\" stroke=\\"#d4dfe5\\" stroke-width=\\"18\\" stroke-linecap=\\"round\\"/><path d=\\"M170 276v-92M256 276v112M342 276v-132\\" stroke=\\"#218a7a\\" stroke-width=\\"18\\" stroke-linecap=\\"round\\"/><circle cx=\\"170\\" cy=\\"168\\" r=\\"30\\" fill=\\"#218a7a\\"/><circle cx=\\"256\\" cy=\\"404\\" r=\\"30\\" fill=\\"#366af0\\"/><circle cx=\\"342\\" cy=\\"128\\" r=\\"30\\" fill=\\"#f5f7fa\\"/></svg>\\n"}')
for name, content in FILES.items():
    dest = APP_DIR / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content, encoding='utf-8')
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
print('대시보드 준비 완료. 추가 패키지 설치 없음.')


## 2. 대시보드 실행
출력되는 버튼을 누르면 코드 없이 대시보드만 별도 탭에 열립니다.


In [ ]:
# 같은 런타임에서 재실행하면 기존 서버를 종료하고 저장 파일을 다시 읽습니다.
if 'dashboard_server' in globals():
    dashboard_server.shutdown()
    dashboard_server.server_close()
# 새 소스를 받은 뒤 다시 실행해도 이전에 불러온 코드가 남지 않게 한다.
for _name in [m for m in sys.modules if m == 'dashboard' or m.startswith('dashboard.')]:
    del sys.modules[_name]
from dashboard.server import start_server
from dashboard.colab import private_link, show_link

dashboard_server = start_server(port=0, data_path=APP_DIR/'data/private/dashboard.json')
show_link(private_link(dashboard_server), '별도 탭에서 대시보드 열기')
print('코랩 전용 링크입니다. Google 계정으로 로그인한 현재 사용자를 위한 접속 방식입니다.')
print('런타임이 종료되면 서버도 종료됩니다. 대시보드 상단에서 백업 저장을 눌러주세요.')


## 3. 다른 사람에게 공유하는 임시 링크 (선택)
기본값은 꺼짐입니다. 공유가 필요하면 PUBLIC_LINK를 True로 바꾸세요. 링크 소지자 모두 같은 편집 권한을 갖는 초기 버전입니다.


In [ ]:
# 공개 링크가 필요할 때만 아래 값을 True로 바꾸고 이 셀을 실행하세요.
# 설치는 내 PC가 아니라 Colab의 임시 실행 환경에서만 이루어집니다.
PUBLIC_LINK = False
if PUBLIC_LINK:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'gradio==5.49.1', 'huggingface-hub<1.0'], check=True)
    if 'public_demo' in globals():
        public_demo.close()
    from dashboard.colab import public_link
    public_demo, url = public_link(dashboard_server)
    show_link(url, '공유 링크에서 대시보드 열기')
    print('이 전용 링크를 가진 사람은 같은 데이터를 열람·수정할 수 있습니다.')
    print('링크는 임시이며 코랩 런타임 종료 또는 터널 만료 시 사용할 수 없습니다.')
else:
    print('공개 링크를 만들지 않았습니다. 위의 코랩 전용 링크를 사용하세요.')
